# Multi-seed reliability study — BraTS 2020 activation-function ablation

The main study trains each activation function once with `seed = 42`. This notebook repeats the training for the four activation functions discussed most in the paper across **three seeds** and reports every metric as mean ± SD across seeds, then tests whether the necrotic-core (NCR) Dice difference between Swish and the ReLU baseline reproduces from seed to seed.

| Setting | Value |
|---|---|
| Seeds | **42, 1337, 2025** (42 = main study) |
| Activations | **ReLU** (baseline), **Swish**, **PReLU**, **TanhExp** |
| Trainings | 4 activations × 3 seeds = **12 runs**, 100 epochs each |
| Everything else | identical to the main study (model, loss, optimiser, split, preprocessing) |

The pipeline is unchanged: the same `ImprovedUNet3D(4, 4, act)`, the same unweighted cross-entropy + soft-Dice loss, AdamW (lr 3e-4) with cosine annealing, the same 80/20 volume split and the same z-score-normalised 128³ inputs. Only the seed changes between runs.

Each run is checkpointed and resumable: re-run the training cell after a disconnect and it continues where it stopped. Outputs are the CSV files archived in `results/multiseed/`.

## 1 · Mount Drive and configure
Reuses the preprocessed 128³ volumes from the main study (no re-preprocessing). Results are written to a separate folder so the main-study outputs are never overwritten.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

# --- Paths (reuse existing preprocessed data; write to a NEW results folder) ---
DATA_DIR     = "/content/drive/MyDrive/BraTS2020_Preprocessed_128"   # already-preprocessed 128^3 x 4ch volumes
RESULTS_ROOT = "/content/drive/MyDrive/BraTS_MultiSeed_Results"      # new — keeps original results intact
os.makedirs(RESULTS_ROOT, exist_ok=True)

# --- Experiment configuration ---
SEEDS       = [42, 1337, 2025]
ACTIVATIONS = ["relu", "swish", "prelu", "tanhexp"]   # the four activations examined in the follow-up studies
EPOCHS      = 100          # as in the main study
BATCH_SIZE  = 2
LR          = 3e-4

assert os.path.exists(DATA_DIR), f"Preprocessed data not found at {DATA_DIR}. Run the original preprocessing cell first."
print("Drive mounted.")
print(f"   Data:    {DATA_DIR}")
print(f"   Results: {RESULTS_ROOT}")
print(f"   Plan:    {len(ACTIVATIONS)} activations x {len(SEEDS)} seeds = {len(ACTIVATIONS)*len(SEEDS)} trainings")

## 2 · Model, activations, dataset, seeding
Copied verbatim from `src/train.py` / `src/evaluate.py` so that the architecture and data handling are identical. `seed_everything` is extended with a DataLoader generator and `worker_init_fn` so that the seed also controls data-shuffling order.

In [ ]:
import time, random, glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd
from tqdm.auto import tqdm

# ---------- Reproducibility ----------
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _worker_init_fn(worker_id):
    # ensure each DataLoader worker is seeded deterministically off the global seed
    s = torch.initial_seed() % (2**32)
    np.random.seed(s); random.seed(s)

# ---------- Activations (identical to original) ----------
class Mish(nn.Module):
    def forward(self, x): return x * torch.tanh(F.softplus(x))
class ELiSH(nn.Module):
    def forward(self, x): return F.elu(x) * torch.sigmoid(x)
class HardELiSH(nn.Module):
    def forward(self, x): return F.elu(x) * F.hardsigmoid(x)
class Logish(nn.Module):
    def forward(self, x): return x * torch.log(1 + torch.sigmoid(x))
class Smish(nn.Module):
    def forward(self, x): return x * torch.tanh(torch.log(1 + torch.sigmoid(x)))
class TanhExp(nn.Module):
    def forward(self, x): return x * torch.tanh(torch.exp(torch.clamp(x, max=20)))

def get_activation(name):
    name = name.lower()
    if name == 'relu': return nn.ReLU(inplace=True)
    if name == 'leaky_relu': return nn.LeakyReLU(0.01, inplace=True)
    if name == 'prelu': return nn.PReLU()
    if name == 'elu': return nn.ELU(inplace=True)
    if name == 'gelu': return nn.GELU()
    if name == 'swish': return nn.SiLU(inplace=True)
    if name == 'mish': return Mish()
    if name == 'elish': return ELiSH()
    if name == 'hard_elish': return HardELiSH()
    if name == 'logish': return Logish()
    if name == 'smish': return Smish()
    if name == 'tanhexp': return TanhExp()
    raise ValueError(f"Activation {name} not supported.")

# ---------- Model (identical to original ImprovedUNet3D) ----------
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, act_name):
        super().__init__()
        self.conv1 = nn.Conv3d(in_c, out_c, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(out_c)
        self.act = get_activation(act_name)
        self.conv2 = nn.Conv3d(out_c, out_c, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(out_c)
        self.skip = nn.Conv3d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
    def forward(self, x):
        return self.act(self.bn2(self.conv2(self.act(self.bn1(self.conv1(x))))) + self.skip(x))

class ImprovedUNet3D(nn.Module):
    def __init__(self, in_c, out_c, act_name):
        super().__init__()
        self.enc1 = ResidualBlock(in_c, 32, act_name); self.pool = nn.MaxPool3d(2)
        self.enc2 = ResidualBlock(32, 64, act_name)
        self.enc3 = ResidualBlock(64, 128, act_name)
        self.enc4 = ResidualBlock(128, 256, act_name)
        self.bottleneck = ResidualBlock(256, 512, act_name)
        self.up4 = nn.ConvTranspose3d(512, 256, 2, 2); self.dec4 = ResidualBlock(512, 256, act_name)
        self.up3 = nn.ConvTranspose3d(256, 128, 2, 2); self.dec3 = ResidualBlock(256, 128, act_name)
        self.up2 = nn.ConvTranspose3d(128, 64, 2, 2);  self.dec2 = ResidualBlock(128, 64, act_name)
        self.up1 = nn.ConvTranspose3d(64, 32, 2, 2);   self.dec1 = ResidualBlock(64, 32, act_name)
        self.out = nn.Conv3d(32, out_c, 1)
    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4=self.dec4(torch.cat([self.up4(b), e4], 1)); d3=self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2=self.dec2(torch.cat([self.up2(d3), e2], 1)); d1=self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)

# ---------- Dataset (returns filename too, for patient-paired stats) ----------
class BraTSDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir; self.mask_dir = mask_dir
        self.files = sorted([f for f in os.listdir(img_dir) if f.endswith('.npy')])
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img  = np.load(os.path.join(self.img_dir,  self.files[idx])).astype(np.float32)
        mask = np.load(os.path.join(self.mask_dir, self.files[idx])).astype(np.longlong)
        return (torch.from_numpy(img).permute(3, 2, 0, 1),
                torch.from_numpy(mask).permute(2, 0, 1),
                self.files[idx])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Components defined. Device:", device)
print("   Sanity-check param count (should be ~22.93M):",
      f"{sum(p.numel() for p in ImprovedUNet3D(4,4,'relu').parameters())/1e6:.2f}M")

## 3 · Training (one run per activation × seed)
Same recipe as the main study: combined loss = unweighted cross-entropy + soft Dice, AdamW (lr 3e-4), cosine annealing over 100 epochs, AMP mixed precision. The best checkpoint per run is the epoch with the highest mean per-class validation Dice — the same selection rule as `src/train.py`. Each run saves `best_model.pth`, `latest.pth` (for resume) and `log.csv` under `…/{ACT}_seed{SEED}/`.

In [ ]:
def train_one(act_name, seed, epochs=EPOCHS):
    # Train a single ImprovedUNet3D(act_name) with a given seed. Resumable.
    run_name = f"{act_name}_seed{seed}"
    save_dir = os.path.join(RESULTS_ROOT, run_name)
    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, "latest.pth")

    seed_everything(seed)                      # <-- the only thing that changes between runs

    model     = ImprovedUNet3D(4, 4, act_name).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    scaler    = torch.amp.GradScaler('cuda')
    ce_loss   = nn.CrossEntropyLoss()          # UNWEIGHTED (matches the code used for reported results)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    start_epoch, history = 0, []
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['opt'])
        start_epoch = ck['epoch']; history = ck.get('history', [])
        if 'scheduler' in ck: scheduler.load_state_dict(ck['scheduler'])
        else:
            for _ in range(start_epoch): scheduler.step()
        if start_epoch >= epochs:
            print(f"   already complete: {run_name} ({start_epoch} epochs)."); return save_dir
        print(f"   Resuming {run_name} from epoch {start_epoch}.")

    # Deterministic data ordering tied to the seed
    g = torch.Generator(); g.manual_seed(seed)
    train_dl = DataLoader(BraTSDataset(f"{DATA_DIR}/train/images", f"{DATA_DIR}/train/masks"),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True,
                          generator=g, worker_init_fn=_worker_init_fn)
    val_dl   = DataLoader(BraTSDataset(f"{DATA_DIR}/val/images",  f"{DATA_DIR}/val/masks"),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    for epoch in range(start_epoch, epochs):
        torch.cuda.reset_peak_memory_stats(); t0 = time.time()

        # ---- train ----
        model.train(); t_loss = 0.0
        for x, y, _ in tqdm(train_dl, desc=f"{run_name} | Ep {epoch+1} train", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                pred   = model(x)
                p_soft = F.softmax(pred, 1)
                y_oh   = F.one_hot(y, 4).permute(0,4,1,2,3).float()
                inter  = (p_soft * y_oh).sum((2,3,4))
                union  = p_soft.sum((2,3,4)) + y_oh.sum((2,3,4))
                dice_loss = 1 - ((2*inter + 1e-5)/(union + 1e-5)).mean()
                loss = ce_loss(pred, y) + dice_loss          # CE + soft-Dice
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            t_loss += loss.item()
        train_loss = t_loss / len(train_dl)

        # ---- validate ----
        model.eval(); v_loss = 0.0; dices = np.zeros(4)
        with torch.no_grad():
            for x, y, _ in tqdm(val_dl, desc=f"{run_name} | Ep {epoch+1} val", leave=False):
                x, y = x.to(device), y.to(device)
                with torch.amp.autocast('cuda'):
                    pred   = model(x)
                    p_soft = F.softmax(pred, 1)
                    y_oh   = F.one_hot(y, 4).permute(0,4,1,2,3).float()
                    inter  = (p_soft * y_oh).sum((2,3,4))
                    union  = p_soft.sum((2,3,4)) + y_oh.sum((2,3,4))
                    dice_loss = 1 - ((2*inter + 1e-5)/(union + 1e-5)).mean()
                    loss = ce_loss(pred, y) + dice_loss
                v_loss += loss.item()
                p_cls = pred.argmax(1)
                for c in range(4):
                    p = (p_cls==c).float(); t = (y==c).float()
                    tp = (p*t).sum(); fp = (p*(1-t)).sum(); fn = ((1-p)*t).sum()
                    dices[c] += ((2*tp + 1e-5)/(2*tp + fp + fn + 1e-5)).item()
        val_loss = v_loss/len(val_dl); dices /= len(val_dl)
        mean_dice = dices[1:].mean()       # mean over NCR/ED/ET classes (same as original)
        dur = time.time()-t0; mem = torch.cuda.max_memory_allocated()/1e9
        print(f"   {run_name} Ep {epoch+1}/{epochs}: val_loss={val_loss:.4f} mean_dice={mean_dice:.4f} ({dur:.0f}s, {mem:.1f}GB)")

        history.append([epoch+1, train_loss, val_loss, mean_dice, *dices, dur, mem])
        torch.save({'epoch':epoch+1,'model':model.state_dict(),'opt':optimizer.state_dict(),
                    'scheduler':scheduler.state_dict(),'history':history}, ckpt_path)
        if mean_dice >= max(h[3] for h in history):
            torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pth"))
        pd.DataFrame(history, columns=['epoch','train_loss','val_loss','mean_dice',
                     'dice_bg','dice_ncr','dice_ed','dice_et','time','mem_gb']
                    ).to_csv(os.path.join(save_dir, "log.csv"), index=False)
        scheduler.step()

    print(f"   Finished {run_name}.")
    return save_dir

print("train_one() ready.")

In [ ]:
# ============================================================
#  RUN ALL 12 TRAININGS (4 activations x 3 seeds), resumable.
#  Re-run this cell after any disconnect — completed runs are skipped.
# ============================================================
import sys
# Silence the harmless "can only test a child process" messages that PyTorch's
# DataLoader emits during worker cleanup in Colab. Genuine errors still print.
def _quiet_unraisable(unraisable):
    exc = unraisable.exc_value
    if isinstance(exc, AssertionError) and "child process" in str(exc):
        return
    sys.__unraisablehook__(unraisable)
sys.unraisablehook = _quiet_unraisable

for seed in SEEDS:
    for act in ACTIVATIONS:
        print(f"\n{'='*60}\n  {act.upper()}  |  seed {seed}\n{'='*60}")
        train_one(act, seed, EPOCHS)
print("\nAll multi-seed trainings complete (or resumed to completion).")

## 4 · Evaluation — per-patient, per-region metrics
Identical metric engine to `src/evaluate.py`: Dice, 95 % Hausdorff distance (empty-vs-non-empty penalised at **373 mm**), sensitivity and precision for the five regions **ET, TC, WT, NCR, ED**. Writes one row per (activation, seed, patient) to `raw_predictions_multiseed.csv` — the multi-seed analogue of `raw_predictions.csv`, with an added `Seed` column.

In [ ]:
from scipy.ndimage import distance_transform_edt as distance

def compute_metrics(pred, gt):
    pred = pred.astype(bool); gt = gt.astype(bool)
    if pred.sum()==0 and gt.sum()==0:
        return 1.0, 0.0, 1.0, 1.0
    inter = (pred & gt).sum()
    dice = (2.*inter)/(pred.sum()+gt.sum()+1e-5)
    sens = inter/(gt.sum()+1e-5)
    prec = inter/(pred.sum()+1e-5)
    try:
        if pred.sum()>0 and gt.sum()>0:
            dt_gt = distance(1-gt); dt_pred = distance(1-pred)
            hd95 = np.percentile(np.hstack([dt_gt[pred], dt_pred[gt]]), 95)
        else:
            hd95 = 373.0
    except Exception:
        hd95 = 373.0
    return dice, hd95, sens, prec

RAW_MS_PATH = os.path.join(RESULTS_ROOT, "raw_predictions_multiseed.csv")

def evaluate_all():
    val_dl = DataLoader(BraTSDataset(f"{DATA_DIR}/val/images", f"{DATA_DIR}/val/masks"),
                        batch_size=1, shuffle=False, num_workers=2)
    if os.path.exists(RAW_MS_PATH):
        done = set(map(tuple, pd.read_csv(RAW_MS_PATH)[['Activation','Seed','Patient']].values.tolist()))
    else:
        done = set()

    for seed in SEEDS:
        for act in ACTIVATIONS:
            run_name = f"{act}_seed{seed}"
            mfile = os.path.join(RESULTS_ROOT, run_name, "best_model.pth")
            if not os.path.exists(mfile):
                print(f"   Skipping {run_name} (no best_model.pth yet)"); continue
            model = ImprovedUNet3D(4,4,act).to(device)
            model.load_state_dict(torch.load(mfile, map_location=device)); model.eval()
            print(f"Evaluating {run_name}")
            buf = []
            with torch.no_grad():
                for x, y, fname in tqdm(val_dl, leave=False):
                    pid = fname[0]
                    if (act, seed, pid) in done: continue
                    x = x.to(device); y_np = y.numpy()[0]
                    with torch.amp.autocast('cuda'):
                        p_cls = model(x).argmax(1).cpu().numpy()[0]
                    regions = {
                        'WT': (p_cls>0,               y_np>0),
                        'TC': ((p_cls==1)|(p_cls==3), (y_np==1)|(y_np==3)),
                        'ET': (p_cls==3,              y_np==3),
                        'NCR':(p_cls==1,              y_np==1),
                        'ED': (p_cls==2,              y_np==2),
                    }
                    row = {'Activation':act, 'Seed':seed, 'Patient':pid}
                    for rn,(pm,tm) in regions.items():
                        d,h,s,p = compute_metrics(pm.astype(np.uint8), tm.astype(np.uint8))
                        row[f'Dice_{rn}']=d; row[f'HD95_{rn}']=h; row[f'Sens_{rn}']=s; row[f'Prec_{rn}']=p
                    buf.append(row)
                    if len(buf)>=10:
                        pd.DataFrame(buf).to_csv(RAW_MS_PATH, mode='a',
                                                 header=not os.path.exists(RAW_MS_PATH), index=False)
                        buf=[]
            if buf:
                pd.DataFrame(buf).to_csv(RAW_MS_PATH, mode='a',
                                         header=not os.path.exists(RAW_MS_PATH), index=False)
    print(f"\nPer-patient results saved to: {RAW_MS_PATH}")

evaluate_all()

## 5 · Aggregate across seeds and reproducibility test
Three files are written:

1. **`per_seed_summary.csv`** — mean over patients for each (activation, seed).
2. **`across_seed_summary.csv`** — mean ± SD across the 3 seeds for every activation × metric (Table 11 of the paper).
3. **`seed_significance.csv`** — within each seed, a patient-paired **Wilcoxon signed-rank** test of each activation against ReLU (the same test as the main study).

The cell also prints a plain-language summary of the Swish-vs-ReLU NCR Dice comparison across seeds.

In [ ]:
from scipy.stats import wilcoxon

df = pd.read_csv(RAW_MS_PATH)
REGIONS = ['ET','TC','WT','NCR','ED']
METRICS = ['Dice','HD95','Sens','Prec']
metric_cols = [f"{m}_{r}" for m in METRICS for r in REGIONS]

# 1) per-seed means (mean over patients)
per_seed = (df.groupby(['Activation','Seed'])[metric_cols].mean().reset_index())
per_seed.to_csv(os.path.join(RESULTS_ROOT,"per_seed_summary.csv"), index=False)

# 2) across-seed mean +/- SD (over the 3 per-seed means)
rows = []
for act in ACTIVATIONS:
    sub = per_seed[per_seed['Activation']==act]
    r = {'Activation':act, 'n_seeds':sub['Seed'].nunique()}
    for c in metric_cols:
        r[f'{c}_mean'] = sub[c].mean()
        r[f'{c}_sd']   = sub[c].std(ddof=1)
    rows.append(r)
across = pd.DataFrame(rows)
across.to_csv(os.path.join(RESULTS_ROOT,"across_seed_summary.csv"), index=False)

# 3) per-seed Wilcoxon vs ReLU (patient-paired), each metric/region
sig_rows = []
for seed in SEEDS:
    dseed = df[df['Seed']==seed]
    base  = dseed[dseed['Activation']=='relu']
    for act in ACTIVATIONS:
        if act=='relu': continue
        cur = dseed[dseed['Activation']==act]
        for c in metric_cols:
            m = pd.merge(cur[['Patient',c]], base[['Patient',c]], on='Patient', suffixes=('','_base'))
            p = np.nan
            if len(m)>0 and not (m[c]==m[f'{c}_base']).all():
                try: _, p = wilcoxon(m[c], m[f'{c}_base'])
                except Exception: p = np.nan
            sig_rows.append({'Seed':seed,'Activation':act,'Metric':c,'p_value':p,
                             'sig':'**' if (p==p and p<0.01) else ('*' if (p==p and p<0.05) else '')})
sig = pd.DataFrame(sig_rows)
sig.to_csv(os.path.join(RESULTS_ROOT,"seed_significance.csv"), index=False)

# ---- formatted across-seed Dice table ----
def fmt(act, col):
    row = across[across['Activation']==act].iloc[0]
    return f"{row[f'{col}_mean']:.3f} +/- {row[f'{col}_sd']:.3f}"

print("="*78)
print("  ACROSS-SEED DICE  (mean +/- SD over seeds 42/1337/2025)")
print("="*78)
print(f"{'Activation':<10} | " + " | ".join(f"{r:<16}" for r in REGIONS))
print("-"*78)
for act in ACTIVATIONS:
    print(f"{act:<10} | " + " | ".join(f"{fmt(act, f'Dice_{r}'):<16}" for r in REGIONS))

# ---- headline reliability statement (real numbers) ----
print("\n" + "="*78)
print("  HEADLINE FINDING — reproducibility of Swish vs ReLU on Necrotic Core (NCR)")
print("="*78)
sw = across[across['Activation']=='swish'].iloc[0]
rl = across[across['Activation']=='relu'].iloc[0]
delta = sw['Dice_NCR_mean'] - rl['Dice_NCR_mean']
print(f"  Swish NCR Dice : {sw['Dice_NCR_mean']:.3f} +/- {sw['Dice_NCR_sd']:.3f}  (across {int(sw['n_seeds'])} seeds)")
print(f"  ReLU  NCR Dice : {rl['Dice_NCR_mean']:.3f} +/- {rl['Dice_NCR_sd']:.3f}")
print(f"  Mean improvement: {delta*100:+.2f} Dice points")
ncr_sig = sig[(sig['Activation']=='swish') & (sig['Metric']=='Dice_NCR')][['Seed','p_value','sig']]
print("  Per-seed Wilcoxon (Swish vs ReLU, NCR Dice):")
for _, rr in ncr_sig.iterrows():
    pv = "nan" if rr['p_value']!=rr['p_value'] else f"{rr['p_value']:.4g}"
    print(f"     seed {int(rr['Seed'])}: p = {pv} {rr['sig']}")

print("\nSaved: per_seed_summary.csv, across_seed_summary.csv, seed_significance.csv")
print(f"   in {RESULTS_ROOT}")

## 6 · Outputs
The four CSV files written to `BraTS_MultiSeed_Results/` are archived in this repository under `results/multiseed/`.